# 3a - pandas on real prices

**Time:** 45 minutes  |  **AI: explains** (Gemini may explain an error or a line to you)

**What you will do**
- Meet pandas: a table of five years of real prices, in Python.
- Rebase, compute returns, volatility, correlation, year-by-year and rolling risk.
- Read a `KeyError`, then send a result to Excel.
- For each cell: write your prediction first, then run it, then compare.

How to work: click a cell, press Shift+Enter to run it. Run the cells **one at a time, in order**. Cell 9 fails on purpose, so do not use "Run all".

## The story

You join the TMT coverage team. The MD wants a risk table for five mega-caps against the S&P 500: Apple, Microsoft, Alphabet, Amazon and Meta.

Today you read the data. In 3b you build the table.

The file `peers.xlsx` holds the **adjusted close** (the price corrected for dividends and splits) of each stock, every trading day from 1 November 2021 to 13 November 2026. The S&P 500 is the column `SP500`. It is a Yahoo Finance snapshot.

## Cell 1 - Load five years of prices

`import pandas as pd` loads the pandas toolbox and calls it `pd`. `pd.read_excel(...)` reads an Excel file into a **DataFrame**: a table, like an Excel sheet. `index_col="Date"` says: the column `Date` is the **index**, the labels of the rows.

`DATA` is the address of the data folder in the course repo.

`.shape` gives (rows, columns). `.head()` shows the first 5 rows.

**Your prediction:** how many rows? One per trading day, for five years. About 250 a year...

In [ ]:
import pandas as pd

DATA = "https://raw.githubusercontent.com/niparis/gem-python-finance/main/data/"
prices = pd.read_excel(DATA + "peers.xlsx", index_col="Date")

print(prices.shape)    # (rows, columns)
prices.head()

## Cell 2 - One stock, one day

- A **column**: `prices["AAPL"]` is the Apple prices, all days.
- A **row by label**: `prices.loc["2026-11-13"]` is every stock on that date.
- A **row by position**: `prices.iloc[0]` is the first row. Positions start at 0, like in a list.

**Your prediction:** what is the difference between `prices.loc["2026-11-13"]` and `prices.iloc[-1]`?

In [ ]:
print(prices["AAPL"].head(3))       # a column: one stock, all days
print()
print(prices.loc["2026-11-13"])     # a row by label: all stocks, one day
print()
print(prices.iloc[0])               # a row by position: the first day

## Cell 3 - Who did best? Rebase to 100

Prices are in different units: Apple 145, the S&P 500 4,613. To compare them, **rebase**: divide every row by the first row, times 100. Every stock starts at 100. One line, six series: pandas divides each column by its own first value.

`.plot()` draws one line per column.

**Your prediction:** which stock ends highest? Which ends lowest? Write your ranking here: ...

In [ ]:
rebased = prices / prices.iloc[0] * 100      # every stock starts at 100

print(rebased.iloc[-1].round(0))             # the last row: where each stock ends
rebased.plot()                               # one line per column

## Cell 4 - Daily returns

A **return** is the change from one day to the next: `price today / price yesterday - 1`. `.pct_change()` does it for every column. The first row has no yesterday, so it is empty (`NaN`): `.dropna()` removes it.

Two methods in a row is called **chaining**: `prices.pct_change().dropna()`.

**Your prediction:** how many rows will `returns` have, compared with `prices`?

In [ ]:
returns = prices.pct_change().dropna()

print(returns.shape)
returns.head(3)

## Cell 5 - Annual volatility: Session 0 slide 9, now readable

Volatility is the standard deviation of the returns. `returns.std()` gives one number per column. A daily number is annualized by multiplying by the square root of the number of returns in a year: `252 ** 0.5`. Prices exist only on trading days, so the year has about 252 returns.

You should get Apple **28.1%**, Meta **45.3%**, S&P 500 **17.1%**.

**Your prediction:** which of the six is the most volatile? Which is the least?

In [ ]:
# annualized volatility of each peer
vol = returns.std() * 252 ** 0.5

print(vol.round(3))
print(f"Apple: {vol['AAPL']:.1%}")

## Cell 6 - Year by year

`.resample("YE")` groups the rows by year (`YE` = year end). `.last()` keeps the last price of each year. `.pct_change()` then gives the return from one year-end to the next.

Careful: the first year has nothing before it (`NaN`), and the last row is 2026 up to 13 November only.

You should get Meta **-64%** in 2022 and **+194%** in 2023.

**Your prediction:** was the S&P 500 up or down in 2022? Was Apple better or worse than the index?

In [ ]:
yearly = prices.resample("YE").last().pct_change()

print(yearly.round(2))
print()
print(yearly.loc["2022":"2023", "META"].round(3))

## Cell 7 - Do they move together?

`returns.corr()` gives a table of correlations, every column against every column. A correlation is between -1 and 1. A stock against itself is always 1.

You should get Apple / S&P 500 **0.71**.

**Your prediction:** which is higher, the correlation of Apple with Microsoft, or of Apple with the S&P 500?

In [ ]:
corr = returns.corr()

print(corr.round(2))
print()
print(f"Apple / S&P 500: {corr.loc['AAPL', 'SP500']:.2f}")

## Cell 8 - Risk changes over time

`.rolling(63)` looks at the last 63 rows (about three months of trading), moving one day at a time. `.std()` is then computed on each window. Multiply by `252 ** 0.5` as before: you get a volatility for every day.

You should get Apple between **15% and 52%**.

**Your prediction:** was Apple's risk about 28% all the time? Where is the chart high?

In [ ]:
rolling = returns.rolling(63).std() * 252 ** 0.5

print(f"Apple, lowest:  {rolling['AAPL'].min():.1%}")
print(f"Apple, highest: {rolling['AAPL'].max():.1%}")
rolling["AAPL"].plot()

## Cell 9 - A typo

**This cell fails on purpose.** Read the message: **last line first, then the arrow.**

1. The last line says the type of error and what went wrong.
2. The arrow `---->` shows the line that failed.
3. Ask: which value is not what I think it is?

**Your prediction:** what will the last line say? What is Python looking for?

(Stuck? Ask Gemini to *explain* the error. Then fix it yourself.)

In [ ]:
prices["Apple"]

Fix it yourself: use the column name that exists. Look at the names with `prices.columns`.

In [ ]:
# YOUR CODE HERE: print the column names, then the last Apple price, with the right name

## Cell 10 - Send it to the MD

`.to_excel("vol.xlsx")` writes the table to an Excel file in the Colab folder. In Colab the second part downloads it to your computer. (Outside Colab it just says so.)

**Your prediction:** open the file in Excel: how many columns will you see?

In [ ]:
vol.to_excel("vol.xlsx")

try:
    from google.colab import files
    files.download("vol.xlsx")
except ImportError:
    print("Not in Colab: the file vol.xlsx is in the current folder.")

## What you learned

**Python**
- **Import**: `import pandas as pd`. **DataFrame**: `pd.read_excel("peers.xlsx", index_col="Date")`, a table with an **index** (the dates), columns and rows.
- **Column**: `prices["AAPL"]`. **Row by label**: `prices.loc["2026-11-13"]`. **Row by position**: `prices.iloc[0]`.
- **Method**: `.pct_change().dropna()`, an action on the table, chained one after the other.
- **Statistic per column**: `returns.std()`, `returns.corr()`.
- **Resample**: `.resample("YE").last()`. **Rolling window**: `.rolling(63).std()`.
- **Chart**: `.plot()`. **Export**: `.to_excel("vol.xlsx")`.
- **Error type**: `KeyError: 'Apple'`, no column with that name. Last line first.

**Finance**
- Rebase to 100 to compare stocks with different prices. Apple 245, the S&P 500 164.
- Volatility is `returns.std() * 252 ** 0.5`: 252 returns a year, so 252, not 365.
- One volatility number hides how risk moves: Apple's 63-day volatility went from 15% to 52%. Meta lost 64% in 2022 and gained 194% in 2023.
- Correlation of a stock with the index: Apple / S&P 500 0.71.

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right